# AIR — all additional experiments, unattended

Runs every experiment that strengthens the paper, one after another, on a single GPU (tested setup: RTX 4090, 24 GB).

**How to use**
1. Clone or copy the AIR repository to the GPU machine (this notebook needs the paper's saved results in `results/`).
2. Open this notebook from the repository (`experiments/` or the repository root), edit the **Settings** cell if needed.
3. **Kernel → Restart & Run All.** The first cells install missing packages and ask for your Hugging Face token
   *immediately*; after that nothing waits for input, so you can leave the machine.

**Safe to leave running**
* Each experiment runs inside a crash guard: if one fails, the full error is written to `results/run_all_log.txt`
  and the notebook continues with the next experiment.
* Every experiment saves after each document. If the machine restarts, **Run All** again: finished work is skipped.
* The final cells print a status table and write paper-ready tables to `results/summary/`.

**Approximate GPU time (RTX 4090), in run order:** memory scaling 1 h · whole-document baselines 0.5 h ·
ConflictQA seeds 4.5 h · de-biased StrategyQA 2 h · long-document AUC 1.5 h · StrategyQA seeds 3 h ·
rank ablation 5.5 h · large judge 1 h · second base model 3 h — about **22 hours** with everything enabled.

The library code below is generated from the repository files by `experiments/build_notebook.py`; edit the
`.py` files and regenerate rather than editing those cells here.

## Settings

In [ ]:
# ======================= Settings: edit, then Kernel -> Restart & Run All =======================
RUN = {                                # set False to skip an experiment
    "memory_scaling": True,            # peak memory / time vs. length, 2k-128k tokens        ~1 h
    "baselines_conflictqa": True,      # LLM judge (Gemma) + retrieve-then-NLI + NLI models     ~0.5 h
    "seeds_conflictqa": True,          # AIR + ICL on ConflictQA, extra seeds                  ~2.2 h per seed
    "strategyqa_debiased": True,       # fixed StrategyQA benchmark + its baselines             ~2 h
    "auc_long_documents": True,        # conflict-detection AUC on 8k-token documents          ~1.5 h
    "seeds_strategyqa": True,          # AIR + ICL on StrategyQA, extra seeds                  ~1.5 h per seed
    "rank_ablation": True,             # r in {1,2,4,8,16}, steps in {3,5,10,20}                ~5.5 h
    "baselines_large_judge": True,     # 7B long-context judge in 4-bit (installs bitsandbytes) ~1 h
    "second_model": True,              # AIR + ICL with a second base model on ConflictQA       ~3 h
    "summary": True,                   # paper-ready tables (CPU)                               minutes
}
EXTRA_SEEDS = [43, 44]
MEMORY_LENGTHS = [2048, 4096, 8192, 16384, 32768, 65536, 131072]
AUC_LENGTH, AUC_DOCS = 8192, 40
RANK_N_DOCS = 100                      # 60 halves the rank-ablation time
LARGE_JUDGE = "Qwen/Qwen2.5-7B-Instruct"
SECOND_MODEL = "Qwen/Qwen2.5-3B-Instruct"
INSTALL_MISSING_PACKAGES = True
SHUTDOWN_WHEN_DONE = False             # True powers the machine off at the end (useful on rented GPUs)

## Setup: packages, Hugging Face login, GPU check

In [ ]:
import datetime, importlib.util, json, os, subprocess, sys, time, traceback
from pathlib import Path

# Repository root: the folder that contains results/ with the paper's saved results.
ROOT = next((p for p in [Path.cwd(), Path.cwd().parent] if (p / "results" / "conflictqa").exists()), None)
if ROOT is None:
    raise SystemExit("Open this notebook from inside the AIR repository (results/conflictqa/ was not found).")
(ROOT / "results").mkdir(exist_ok=True)
LOG = ROOT / "results" / "run_all_log.txt"

packages = {"torch": "torch", "transformers": "transformers", "peft": "peft", "accelerate": "accelerate",
            "sentence_transformers": "sentence-transformers", "sklearn": "scikit-learn",
            "matplotlib": "matplotlib", "huggingface_hub": "huggingface_hub"}
if RUN.get("baselines_large_judge"):
    packages["bitsandbytes"] = "bitsandbytes"
missing = [pip for mod, pip in packages.items() if importlib.util.find_spec(mod) is None]
if missing:
    if not INSTALL_MISSING_PACKAGES:
        raise SystemExit(f"Missing packages: {missing}")
    print("installing", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

# Ask for the Hugging Face token NOW, so the run never waits for input later.
from huggingface_hub import get_token, login
if not (os.environ.get("HF_TOKEN") or get_token()):
    from getpass import getpass
    login(token=getpass("Hugging Face token (needs access to google/gemma-2-2b-it): "))

# Fail fast on model access problems instead of hours into the run.
from transformers import AutoTokenizer
needed = ["google/gemma-2-2b-it"]
if RUN.get("baselines_large_judge"):
    needed.append(LARGE_JUDGE)
if RUN.get("second_model"):
    needed.append(SECOND_MODEL)
for name in needed:
    AutoTokenizer.from_pretrained(name)
    print(f"access OK: {name}")

import torch
HAS_GPU = torch.cuda.is_available()
print(f"repository: {ROOT}")
print(f"GPU: {torch.cuda.get_device_name(0) if HAS_GPU else 'none - GPU experiments will be skipped'}")

## Library code
Generated from the repository files; edit the `.py` files and run `python experiments/build_notebook.py` instead of editing these cells.

### `analysis/rebuild_benchmarks.py`
Rebuild the exact 300-document ConflictQA and StrategyQA evaluation sets used in the notebooks.

In [ ]:
import argparse
import json
import random
from pathlib import Path

import numpy as np
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer

CACHE = ROOT / "analysis" / "cache"
SEED = 42
CHUNK_SIZE, CHUNK_OVERLAP, NUM_DISTRACTORS, NUM_SAMPLES = 256, 64, 20, 300


def chunk_document(text, tok, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    token_ids = tok.encode(text, add_special_tokens=False)
    stride = chunk_size - overlap
    chunks, start = [], 0
    while start < len(token_ids):
        end = min(start + chunk_size, len(token_ids))
        chunks.append(tok.decode(token_ids[start:end], skip_special_tokens=True))
        if end >= len(token_ids):
            break
        start += stride
    return chunks


def construct_long_document(doc_a, doc_b, distractors, num_distractors=NUM_DISTRACTORS):
    chosen = random.sample(distractors, min(num_distractors, len(distractors)))
    mid = len(chosen) // 2
    passages = chosen[:mid // 2] + [doc_a] + chosen[mid // 2:mid] + [doc_b] + chosen[mid:]
    return " ".join(passages)


def filtered_pool(probe_query, pool):
    query_terms = [w for w in probe_query.split() if len(w) > 4]
    filtered = [d for d in pool if not any(t.lower() in d.lower() for t in query_terms[:3])]
    return filtered if len(filtered) >= NUM_DISTRACTORS else pool


def build_conflictqa():
    path = hf_hub_download(repo_id="osunlp/ConflictQA", filename="conflictQA-popQA-gpt4.json", repo_type="dataset")
    with open(path, encoding="utf-8") as f:
        first = f.read(1)
        f.seek(0)
        ds = json.load(f) if first == "[" else [json.loads(l) for l in f if l.strip()]

    half, benchmark, cats = NUM_SAMPLES // 2, [], {"cons": 0, "conf": 0}
    for idx, item in enumerate(ds):
        question, ans = item["question"].strip(), item["memory_answer"].strip()
        ev_orig = item["parametric_memory"].strip()
        ev_aligned = item.get("parametric_memory_aligned_evidence", "").strip()
        ev_counter = item["counter_memory"].strip()
        if len(ev_orig) < 40 or len(ev_counter) < 40:
            continue
        claim = f"The answer to the question '{question}' is {ans}."
        if cats["cons"] < half and len(ev_aligned) >= 40 and ev_aligned != ev_orig:
            benchmark.append({"probe_query": claim, "raw_question": question, "doc_a": ev_orig, "doc_b": ev_aligned, "label": 0})
            cats["cons"] += 1
        elif cats["conf"] < half:
            benchmark.append({"probe_query": claim, "raw_question": question, "doc_a": ev_orig, "doc_b": ev_counter, "label": 1})
            cats["conf"] += 1
        if cats["cons"] == half and cats["conf"] == half:
            break
    random.shuffle(benchmark)

    pool = [it.get("parametric_memory", "").strip() for it in list(ds)[-500:]]
    pool = [p for p in pool if len(p) > 50]
    out = []
    for item in benchmark:
        doc = construct_long_document(item["doc_a"], item["doc_b"], filtered_pool(item["probe_query"], pool))
        out.append({**item, "query": f"Is this claim true or false: {item['probe_query']}", "document": doc})
    return out


def build_strategyqa(json_path):
    items = json.load(open(json_path, encoding="utf-8"))
    limit, benchmark = NUM_SAMPLES // 4, []
    cats = {"T_cons": 0, "T_conf": 0, "F_cons": 0, "F_conf": 0}
    a_true = ("Based on established facts, the answer to the question '{q}' is unquestionably True. Multiple reliable "
              "sources have independently verified this claim. The evidence supporting this conclusion is well-documented "
              "and has been confirmed through rigorous analysis by domain experts in the relevant field.")
    a_false = ("Based on established facts, the answer to the question '{q}' is unquestionably False. Multiple reliable "
               "sources have independently verified this conclusion. The evidence against this claim is well-documented and "
               "has been confirmed through rigorous analysis by domain experts in the relevant field.")
    b_true_cons = ("Logical deduction confirms that the premise is accurate and True. Additional supporting evidence has emerged from "
                   "peer-reviewed studies that further corroborate this finding. Independent researchers have reached the same "
                   "conclusion through different methodological approaches and experimental designs.")
    b_false_cons = ("Logical deduction confirms that the premise is accurate and False. Additional supporting evidence has emerged "
                    "from peer-reviewed studies that further corroborate this negative finding. Independent researchers have reached "
                    "the same conclusion through different methodological approaches and experimental designs.")
    b_conf = ("New evidence completely refutes this. The answer to the question '{q}' is absolutely {v}. Recent "
              "investigations have uncovered critical flaws in earlier analyses. Updated research conducted by leading "
              "authorities demonstrates that the original conclusion was based on incomplete or erroneous data.")
    for item in items:
        q, ans = item["question"].strip(), item["answer"]
        if ans is True:
            if cats["T_cons"] < limit:
                benchmark.append({"q": q, "doc_a": a_true.format(q=q), "doc_b": b_true_cons, "label": 0}); cats["T_cons"] += 1
            elif cats["T_conf"] < limit:
                benchmark.append({"q": q, "doc_a": a_true.format(q=q), "doc_b": b_conf.format(q=q, v="False"), "label": 1}); cats["T_conf"] += 1
        elif ans is False:
            if cats["F_cons"] < limit:
                benchmark.append({"q": q, "doc_a": a_false.format(q=q), "doc_b": b_false_cons, "label": 0}); cats["F_cons"] += 1
            elif cats["F_conf"] < limit:
                benchmark.append({"q": q, "doc_a": a_false.format(q=q), "doc_b": b_conf.format(q=q, v="True"), "label": 1}); cats["F_conf"] += 1
        if all(v == limit for v in cats.values()):
            break
    random.shuffle(benchmark)

    pool = []
    for item in items:
        for fact in item.get("facts", []):
            if len(fact) > 5:
                pool.append(f"Research indicates that {fact[0].lower()}{fact[1:]} This has been confirmed by multiple studies and is "
                            f"widely accepted within the academic community. Numerous independent investigations have corroborated "
                            f"this observation through controlled experiments and systematic reviews of the available literature.")
    out = []
    for item in benchmark:
        doc = construct_long_document(item["doc_a"], item["doc_b"], filtered_pool(item["q"], pool))
        out.append({"raw_question": item["q"], "query": f"Is this claim true or false: {item['q']}", "document": doc,
                    "doc_a": item["doc_a"], "doc_b": item["doc_b"], "label": item["label"]})
    return out

### `analysis/gate_analysis.py`
CPU-only re-analysis of AIR's relevance-aware gates from the saved belief states.

In [ ]:
import argparse
import json
import re
from itertools import combinations
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

TOP_K = 3
# Floors used in the paper's main table (collective-comparison cell of the notebooks).
PAPER_RHO = {"lexical": 0.50, "semantic": 0.25, "hybrid": 0.35}
RHO_GRID = np.round(np.arange(0.0, 0.81, 0.05), 2)
STOPWORDS = {
    "what", "where", "when", "which", "who", "whom", "whose", "why", "how",
    "is", "was", "are", "were", "did", "does", "do", "has", "have", "had",
    "the", "a", "an", "in", "on", "at", "of", "for", "to", "from", "by", "with", "about",
}
# The StrategyQA notebook extended the stopword list.
STRATEGYQA_EXTRA = {"can", "could", "would", "should", "will", "more", "than", "most"}


def lexical_relevance(question, chunk, stopwords):
    q = [t.lower() for t in re.findall(r"\w+", question) if t.lower() not in stopwords and len(t) > 2]
    if not q:
        return 1.0
    c = chunk.lower()
    return sum(1 for t in q if t in c) / len(q)


def jsd(p, q, eps=1e-10):
    p, q = np.clip(p, eps, None), np.clip(q, eps, None)
    m = 0.5 * (p + q)
    return float(0.5 * (p * np.log(p / m)).sum() + 0.5 * (q * np.log(q / m)).sum())


def gated_score(beliefs, confs, rel, rho):
    """Relevance-aware gate as in the notebooks: returns (score, abstained, insufficient)."""
    valid = [i for i, r in enumerate(rel) if r >= rho]
    if len(valid) < 2:
        return 0.0, True, True
    top = sorted(valid, key=lambda i: confs[i] * rel[i], reverse=True)[:TOP_K]
    score = max(jsd(beliefs[a], beliefs[b]) for a, b in combinations(top, 2))
    return score, False, len(valid) < TOP_K


def scores_for(docs, rels, rho):
    out = [gated_score(d["beliefs"], d["confs"], r, rho) for d, r in zip(docs, rels)]
    return (np.array([o[0] for o in out]), np.array([o[1] for o in out]), np.array([o[2] for o in out]))


def cv_auc(y, docs, rels, repeats, seed=0):
    """Nested selection of rho: pick rho on 4 folds by AUC, score the held-out fold, pool, compute AUC.

    Also returns the pooled held-out scores of the first repeat, for the bootstrap test against ICL.
    """
    table = {rho: scores_for(docs, rels, rho)[0] for rho in RHO_GRID}
    aucs, chosen, first = [], [], None
    for r in range(repeats):
        held_out = np.zeros(len(y))
        for train, test in StratifiedKFold(5, shuffle=True, random_state=seed + r).split(np.zeros(len(y)), y):
            best = max(RHO_GRID, key=lambda rho: roc_auc_score(y[train], table[rho][train]))
            chosen.append(float(best))
            held_out[test] = table[best][test]
        aucs.append(roc_auc_score(y, held_out))
        if first is None:
            first = held_out.copy()
    return float(np.mean(aucs)), float(np.std(aucs)), chosen, first


def bootstrap(y, s, base, n, seed=0):
    rng = np.random.default_rng(seed)
    a, d = [], []
    while len(a) < n:
        i = rng.integers(0, len(y), len(y))
        if len(set(y[i])) < 2:
            continue
        x = roc_auc_score(y[i], s[i])
        a.append(x)
        d.append(x - roc_auc_score(y[i], base[i]))
    a, d = np.array(a), np.array(d)
    p_value = 2 * min((d <= 0).mean(), (d >= 0).mean())
    return [float(v) for v in np.percentile(a, [2.5, 97.5])], [float(v) for v in np.percentile(d, [2.5, 97.5])], float(min(p_value, 1.0))

### `air/pipeline.py`
AIR pipeline, extracted unchanged from the experiment notebooks so scripts can reuse it.

In [ ]:
from itertools import combinations

import torch
import torch.nn.functional as F

CHUNK_SIZE = 256
CHUNK_OVERLAP = 64
TOP_K = 3
DISTILL_STEPS = 10
DISTILL_LR = 1e-3
MAX_GRAD_NORM = 1.0


def lora_config(r=2, alpha=None):
    from peft import LoraConfig  # imported lazily so CPU-only analyses do not need peft
    return LoraConfig(
        r=r, lora_alpha=alpha if alpha is not None else 2 * r,
        target_modules=["up_proj", "down_proj", "gate_proj", "q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.0, bias="none", task_type="CAUSAL_LM",
    )


def answer_token_ids(tok, variants=False):
    """[yes_id, no_id] exactly as in the notebooks (Gemma). With variants=True, returns ([yes ids], [no ids])
    covering "Yes"/" Yes" and "No"/" No", which other tokenizers (Llama, Qwen) need after "Answer:"."""
    if not variants:
        return [tok.encode("Yes", add_special_tokens=False)[0], tok.encode("No", add_special_tokens=False)[0]]
    first = lambda s: tok.encode(s, add_special_tokens=False)[0]
    return (sorted({first("Yes"), first(" Yes")}), sorted({first("No"), first(" No")}))


def two_way(last_logits, answer_ids):
    """Belief over (Yes, No) from next-token logits of shape [1, V].

    With plain [yes_id, no_id] this is the notebooks' softmax over the two logits. With id groups it
    sums each answer's token variants (log-sum-exp) before normalizing, which is the same quantity."""
    if isinstance(answer_ids[0], int):
        return torch.softmax(last_logits[:, answer_ids].float(), dim=-1).squeeze(0)
    lp = torch.log_softmax(last_logits.float(), dim=-1)
    pair = torch.stack([torch.logsumexp(lp[:, answer_ids[0]], dim=-1), torch.logsumexp(lp[:, answer_ids[1]], dim=-1)], dim=-1)
    return torch.softmax(pair, dim=-1).squeeze(0)


def chunk_document(text, tok, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    token_ids = tok.encode(text, add_special_tokens=False)
    stride = chunk_size - overlap
    chunks, start = [], 0
    while start < len(token_ids):
        end = min(start + chunk_size, len(token_ids))
        chunks.append(tok.decode(token_ids[start:end], skip_special_tokens=True))
        if end >= len(token_ids):
            break
        start += stride
    return chunks


def format_prompt(doc, query):
    return (
        "Answer the question with a single word: Yes or No. No filler words.\n\n"
        "Document: The human body has 206 bones in the adult skeleton.\n"
        "Question: Is this claim true or false: Do adults have more than 200 bones?\n"
        "Answer: Yes\n\n"
        f"Document: {doc}\nQuestion: {query}\nAnswer:"
    )


def distill_chunk_adapter(model, chunk_text, query, tok, cfg, answer_ids,
                          steps=DISTILL_STEPS, lr=DISTILL_LR, max_grad_norm=MAX_GRAD_NORM):
    """Context-distil one chunk into a fresh LoRA; return (belief over [Yes, No], confidence, unloaded model)."""
    from peft import get_peft_model
    peft_model = get_peft_model(model, cfg)
    params = [p for p in peft_model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr)
    peft_model.train()
    t_ids = tok(format_prompt(chunk_text, query), return_tensors="pt", truncation=True, max_length=512).input_ids.to(peft_model.device)
    s_ids = tok(format_prompt("", query), return_tensors="pt", truncation=True, max_length=128).input_ids.to(peft_model.device)
    for _ in range(steps):
        with peft_model.disable_adapter(), torch.no_grad():
            t_dist = torch.softmax(peft_model(t_ids).logits[:, -1, :].float(), dim=-1)
        s_log = torch.log_softmax(peft_model(s_ids).logits[:, -1, :].float(), dim=-1)
        loss = F.kl_div(s_log, t_dist, reduction="batchmean", log_target=False)
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, max_grad_norm)
        opt.step()
    peft_model.eval()
    with torch.no_grad():
        belief = two_way(peft_model(s_ids).logits[:, -1, :], answer_ids).cpu()
    conf = belief.max().item()
    model = peft_model.unload()
    del peft_model, opt, params
    torch.cuda.empty_cache()
    return belief, conf, model


def compute_jsd(p, q, eps=1e-10):
    p, q = p.clamp(min=eps), q.clamp(min=eps)
    m = 0.5 * (p + q)
    return (0.5 * (p * (p / m).log()).sum() + 0.5 * (q * (q / m).log()).sum()).item()


def confidence_gate(beliefs, confs, top_k=TOP_K):
    if len(beliefs) <= top_k:
        return beliefs, list(range(len(beliefs)))
    idx = sorted(range(len(confs)), key=lambda i: confs[i], reverse=True)[:top_k]
    return [beliefs[i] for i in idx], idx


def max_cross_jsd(states, confs=None):
    if len(states) < 2:
        return 0.0
    best = 0.0
    for i, j in combinations(range(len(states)), 2):
        d = compute_jsd(states[i], states[j])
        if confs is not None:
            d *= confs[i] * confs[j]
        best = max(best, d)
    return best


def air_beliefs(document, query, model, tok, cfg, answer_ids, steps=DISTILL_STEPS):
    """Distil every chunk; return per-chunk beliefs, confidences, chunk texts, and the (unloaded) model."""
    chunks = chunk_document(document, tok)
    beliefs, confs = [], []
    for ch in chunks:
        b, c, model = distill_chunk_adapter(model, ch, query, tok, cfg, answer_ids, steps=steps)
        beliefs.append(b)
        confs.append(c)
    return beliefs, confs, chunks, model


def icl_split_window(document, query, model, tok, answer_ids, eps=1e-10, max_length=2048):
    ids = tok.encode(document, add_special_tokens=False)
    mid = len(ids) // 2
    halves = [tok.decode(ids[:mid], skip_special_tokens=True), tok.decode(ids[mid:], skip_special_tokens=True)]
    dists = []
    with torch.no_grad():
        for h in halves:
            x = tok(format_prompt(h, query), return_tensors="pt", truncation=True, max_length=max_length).input_ids.to(model.device)
            dists.append(two_way(model(x).logits[:, -1, :], answer_ids).cpu().clamp(min=eps))
    return compute_jsd(dists[0], dists[1])

### `experiments/benchmarks.py`
Benchmark builders shared by the experiment scripts.

In [ ]:
import random
import sys
from pathlib import Path

import numpy as np
from huggingface_hub import hf_hub_download


NAMES = ["conflictqa", "strategyqa", "strategyqa_debiased"]

# Several phrasings per role so no fixed wording correlates with the label.
DEBIASED_A = [
    "{facts} Based on these facts, the answer to the question '{q}' is {v}.",
    "{facts} Taken together, this evidence indicates that the answer to the question '{q}' is {v}.",
    "{facts} From this, it follows that the answer to the question '{q}' is {v}.",
]
DEBIASED_B = [
    "A separate analysis of the question '{q}' concludes that the answer is {v}. This assessment draws on independent sources and a review of the relevant literature.",
    "An independent review of the question '{q}' reports that the answer is {v}. The review compiles evidence from several published references.",
    "According to another source that examined the question '{q}', the answer is {v}. That source summarizes findings from multiple studies.",
]


def strategyqa_json():
    local = ROOT.parent.parent / "strategyQA" / "strategyqa_train (1).json"
    if local.exists():
        return str(local)
    return hf_hub_download(repo_id="voidful/StrategyQA", filename="strategyqa_train.json", repo_type="dataset")


def build_strategyqa_debiased(json_path, n=NUM_SAMPLES):
    import json
    items = json.load(open(json_path, encoding="utf-8"))
    limit, bench = n // 4, []
    cats = {"T_cons": 0, "T_conf": 0, "F_cons": 0, "F_conf": 0}
    for item in items:
        q, ans, facts = item["question"].strip(), item["answer"], item.get("facts", [])
        if not isinstance(ans, bool) or not facts:
            continue
        gold, other = ("True", "False") if ans else ("False", "True")
        key = "T" if ans else "F"
        if cats[f"{key}_cons"] < limit:
            label, v_b = 0, gold
            cats[f"{key}_cons"] += 1
        elif cats[f"{key}_conf"] < limit:
            label, v_b = 1, other
            cats[f"{key}_conf"] += 1
        else:
            continue
        doc_a = random.choice(DEBIASED_A).format(facts=" ".join(facts), q=q, v=gold)
        doc_b = random.choice(DEBIASED_B).format(q=q, v=v_b)
        bench.append({"q": q, "doc_a": doc_a, "doc_b": doc_b, "label": label})
        if all(v == limit for v in cats.values()):
            break
    random.shuffle(bench)

    pool = []
    for item in items:
        for fact in item.get("facts", []):
            if len(fact) > 5:
                pool.append(f"Research indicates that {fact[0].lower()}{fact[1:]} This has been confirmed by multiple studies and is "
                            f"widely accepted within the academic community. Numerous independent investigations have corroborated "
                            f"this observation through controlled experiments and systematic reviews of the available literature.")
    out = []
    for item in bench:
        doc = construct_long_document(item["doc_a"], item["doc_b"], filtered_pool(item["q"], pool))
        out.append({"raw_question": item["q"], "query": f"Is this claim true or false: {item['q']}", "document": doc,
                    "doc_a": item["doc_a"], "doc_b": item["doc_b"], "label": item["label"]})
    return out


def build(name, seed=42):
    """Return a list of {query, raw_question, document, doc_a, doc_b, label} for the named benchmark."""
    random.seed(seed)
    np.random.seed(seed)
    if name == "conflictqa":
        return build_conflictqa()
    if name == "strategyqa":
        return build_strategyqa(strategyqa_json())
    if name == "strategyqa_debiased":
        return build_strategyqa_debiased(strategyqa_json())
    raise ValueError(f"unknown benchmark {name}; choose from {NAMES}")


def balanced_subset(docs, n):
    """First n/2 consistent and n/2 conflicting documents, in benchmark order."""
    cons = [d for d in docs if d["label"] == 0][: n // 2]
    conf = [d for d in docs if d["label"] == 1][: n - n // 2]
    keep = {id(d) for d in cons + conf}
    return [d for d in docs if id(d) in keep]

### `experiments/run_air.py`
Shared GPU runner: AIR per-chunk beliefs + split-window ICL on one benchmark configuration.

In [ ]:
import argparse
import json
import re
import sys
import time
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


DEFAULT_MODEL = "google/gemma-2-2b-it"


def run_name(benchmark, seed, model, rank, steps, n_docs):
    short = re.sub(r"[^A-Za-z0-9.]+", "-", model.split("/")[-1]).lower()
    name = f"{benchmark}_seed{seed}_{short}_r{rank}_s{steps}"
    return name + (f"_n{n_docs}" if n_docs else "")


def load_model(name, load_4bit=False):
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kw = {"dtype": torch.bfloat16, "device_map": "cuda"}
    if load_4bit:
        from transformers import BitsAndBytesConfig
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
    model = AutoModelForCausalLM.from_pretrained(name, **kw).eval()
    return tok, model


def adapter_bytes(model, cfg):
    """Memory of one Micro-LoRA adapter for this model and config (trainable LoRA tensors)."""
    from peft import get_peft_model
    m = get_peft_model(model, cfg)
    size = sum(p.numel() * p.element_size() for p in m.parameters() if p.requires_grad)
    model = m.unload()
    return size, model


def run(benchmark="conflictqa", seed=42, model_name=DEFAULT_MODEL, rank=2, steps=10, n_docs=0,
        skip_icl=False, load_4bit=False, out_dir=None, tok_model=None):
    """Run (or resume) one configuration. tok_model=(tok, model) reuses an already-loaded model."""
    out_dir = Path(out_dir or ROOT / "results" / "runs")
    out_dir.mkdir(parents=True, exist_ok=True)
    out = out_dir / f"{run_name(benchmark, seed, model_name, rank, steps, n_docs)}.json"

    docs = build(benchmark, seed)
    if n_docs:
        docs = balanced_subset(docs, n_docs)

    tok, model = tok_model or load_model(model_name, load_4bit)
    torch.manual_seed(seed)
    variants = "gemma" not in model_name.lower()  # Gemma keeps the notebooks' exact answer reading
    answer_ids = answer_token_ids(tok, variants=variants)
    cfg = lora_config(r=rank)

    state = json.load(open(out, encoding="utf-8")) if out.exists() else None
    if state is None:
        size, model = adapter_bytes(model, cfg)
        state = {"config": {"benchmark": benchmark, "seed": seed, "model": model_name, "rank": rank, "alpha": 2 * rank,
                            "steps": steps, "n_docs": len(docs), "answer_ids": answer_ids, "adapter_bytes": size,
                            "gpu": torch.cuda.get_device_name(0)},
                 "records": []}
    done = len(state["records"])
    print(f"{out.name}: {done}/{len(docs)} documents already done")

    for i in range(done, len(docs)):
        d = docs[i]
        t0 = time.time()
        beliefs, confs, chunks, model = air_beliefs(d["document"], d["query"], model, tok, cfg, answer_ids, steps=steps)
        air_seconds = time.time() - t0
        gated, idx = confidence_gate(beliefs, confs)
        rec = {
            "label": d["label"], "query": d["query"], "raw_question": d["raw_question"],
            "num_chunks": len(chunks), "chunks": chunks, "confidences": confs,
            "all_belief_states": [(float(b[0]), float(b[1])) for b in beliefs],
            "air_confidence_only": max_cross_jsd(gated, [confs[j] for j in idx]),
            "air_seconds": air_seconds,
        }
        if not skip_icl:
            rec["icl_split_window"] = icl_split_window(d["document"], d["query"], model, tok, answer_ids)
        state["records"].append(rec)
        json.dump(state, open(out, "w", encoding="utf-8"))
        print(f"  [{i + 1}/{len(docs)}] label={d['label']} chunks={len(chunks)} {air_seconds:.1f}s")
    return out, (tok, model)

### `experiments/scoring.py`
CPU-side scoring shared by summarize.py and baselines_full_document.py.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score


PAPER_RHO = {"lexical": 0.50, "semantic": 0.25, "hybrid": 0.35}
NOTEBOOK_RESULTS = {"conflictqa": ROOT / "results/conflictqa/conflicts_air_results_300.json",
                    "strategyqa": ROOT / "results/strategyqa/strategyqa_air_results_300.json"}
_ST = None


def sentence_model():
    global _ST
    if _ST is None:
        from sentence_transformers import SentenceTransformer
        import torch
        _ST = SentenceTransformer("all-MiniLM-L6-v2", device="cuda" if torch.cuda.is_available() else "cpu")
    return _ST


def stopwords(benchmark):
    return STOPWORDS | (STRATEGYQA_EXTRA if benchmark.startswith("strategyqa") else set())


def relevance(records, benchmark):
    """Lexical, MiniLM-semantic and hybrid relevance of every chunk to the raw question."""
    st, stop = sentence_model(), stopwords(benchmark)
    lex, sem = [], []
    for r in records:
        q = r["raw_question"]
        lex.append([lexical_relevance(q, c, stop) for c in r["chunks"]])
        e = st.encode([q] + r["chunks"], normalize_embeddings=True, show_progress_bar=False)
        sem.append(np.clip(e[0] @ e[1:].T, 0.0, 1.0).tolist())
    hyb = [[0.5 * a + 0.5 * b for a, b in zip(l, s)] for l, s in zip(lex, sem)]
    return {"lexical": lex, "semantic": sem, "hybrid": hyb}


def gate_docs(records):
    return [{"beliefs": [np.array(b, dtype=float) for b in r["all_belief_states"]], "confs": r["confidences"]} for r in records]


def prf(y, s, threshold):
    p, r, f, _ = precision_recall_fscore_support(y, (np.asarray(s) > threshold).astype(int), average="binary", zero_division=0)
    return float(p), float(r), float(f)


def score_run(records, benchmark, n_boot=2000, cv_repeats=20):
    """All metrics for one run: confidence-only, ICL, and each relevance gate at the paper floor and with CV."""
    y = np.array([r["label"] for r in records])
    docs, rels = gate_docs(records), relevance(records, benchmark)
    has_icl = all("icl_split_window" in r for r in records)
    icl = np.array([r["icl_split_window"] for r in records], dtype=float) if has_icl else None
    out = {"n": int(len(y)), "methods": {}}

    def add(name, s, extra=None, threshold=0.15):
        m = {"auc": float(roc_auc_score(y, s))}
        m["precision"], m["recall"], m["f1"] = prf(y, s, threshold)
        if icl is not None and name != "ICL split-window":
            ci, dci, p = bootstrap(y, np.asarray(s), icl, n_boot)
            m.update(ci=ci, diff_vs_icl_ci=dci, p_vs_icl=p)
        m.update(extra or {})
        out["methods"][name] = m

    if icl is not None:
        add("ICL split-window", icl, {"ci": bootstrap(y, icl, icl, n_boot)[0]})
    add("AIR confidence-only", np.array([r["air_confidence_only"] for r in records]))
    for gate, rho in PAPER_RHO.items():
        s, abst, insuff = scores_for(docs, rels[gate], rho)
        cv_mean, cv_std, chosen, cv_scores = cv_auc(y, docs, rels[gate], cv_repeats)
        extra = {"rho": rho, "abstained_consistent": int(abst[y == 0].sum()), "abstained_conflict": int(abst[y == 1].sum()),
                 "insufficient": int(insuff.sum()), "cv_auc_mean": cv_mean, "cv_auc_std": cv_std,
                 "cv_rho_counts": {str(r): chosen.count(r) for r in sorted(set(chosen))}}
        if icl is not None:
            extra["cv_p_vs_icl"] = bootstrap(y, cv_scores, icl, n_boot)[2]
        add(f"AIR {gate}", s, extra)
    return out


def load_notebook_run(benchmark):
    """The paper's original seed-42 Gemma results, as run records (chunks re-derived from the rebuilt documents)."""
    from transformers import AutoTokenizer
    saved = json.load(open(NOTEBOOK_RESULTS[benchmark], encoding="utf-8"))
    docs = build(benchmark, 42)
    tok = AutoTokenizer.from_pretrained("google/gemma-2-2b-it")
    records = []
    for d, meta, s_air, s_icl in zip(docs, saved["metadata"], saved["scores_air"], saved["scores_icl"]):
        chunks = chunk_document(d["document"], tok)
        assert len(chunks) == meta["num_chunks"], "rebuilt document does not match the saved results"
        records.append({"label": d["label"], "raw_question": d["raw_question"], "query": d["query"], "chunks": chunks,
                        "confidences": meta["confidences"], "all_belief_states": meta["all_belief_states"],
                        "air_confidence_only": s_air, "icl_split_window": s_icl, "document": d["document"],
                        "doc_a": d["doc_a"], "doc_b": d["doc_b"]})
    config = {"benchmark": benchmark, "seed": 42, "model": "google/gemma-2-2b-it", "rank": 2, "steps": 10,
              "source": "paper notebooks"}
    return {"config": config, "records": records}

### `experiments/memory_scaling.py`
Experiment 1.1: peak GPU memory and wall-clock time vs. document length (AIR vs. ICL).

In [ ]:
import argparse
import json
import random
import sys
import time
from pathlib import Path

import numpy as np
import torch
from huggingface_hub import hf_hub_download
from sklearn.metrics import roc_auc_score
from transformers import AutoModelForCausalLM, AutoTokenizer


GiB = 1024 ** 3
DEFAULT_LENGTHS = [2048, 4096, 8192, 16384, 32768, 65536, 131072]


def load_conflictqa():
    path = hf_hub_download(repo_id="osunlp/ConflictQA", filename="conflictQA-popQA-gpt4.json", repo_type="dataset")
    with open(path, encoding="utf-8") as f:
        first = f.read(1)
        f.seek(0)
        return json.load(f) if first == "[" else [json.loads(l) for l in f if l.strip()]


def make_items(ds, n, seed):
    """Balanced (consistent / conflicting) ConflictQA items, built exactly like the paper's benchmark."""
    rng = random.Random(seed)
    cons, conf = [], []
    for item in ds:
        q, ans = item["question"].strip(), item["memory_answer"].strip()
        a, al, c = item["parametric_memory"].strip(), item.get("parametric_memory_aligned_evidence", "").strip(), item["counter_memory"].strip()
        if len(a) < 40 or len(c) < 40:
            continue
        query = f"Is this claim true or false: The answer to the question '{q}' is {ans}."
        if len(al) >= 40 and al != a:
            cons.append({"query": query, "raw_question": q, "doc_a": a, "doc_b": al, "label": 0})
        conf.append({"query": query, "raw_question": q, "doc_a": a, "doc_b": c, "label": 1})
    rng.shuffle(cons)
    rng.shuffle(conf)
    items = cons[: n // 2] + conf[: n - n // 2]
    rng.shuffle(items)
    return items


def build_document(item, pool, target_tokens, tok, rng):
    """Two target passages among distractors (5 before d_a, 5 between, rest after), padded to target_tokens."""
    terms = [w.lower() for w in item["query"].split() if len(w) > 4][:5]
    candidates = [p for p in pool if not any(t in p.lower() for t in terms)]
    rng.shuffle(candidates)
    chosen, total, k = [], len(tok.encode(item["doc_a"] + " " + item["doc_b"], add_special_tokens=False)), 0
    while total < target_tokens:
        p = candidates[k % len(candidates)]
        k += 1
        chosen.append(p)
        total += len(tok.encode(p, add_special_tokens=False)) + 1
    before, between = chosen[:5], chosen[5:10]
    after = chosen[10:]
    return " ".join(before + [item["doc_a"]] + between + [item["doc_b"]] + after)


def measure(fn):
    """Run fn() and return (result, additional peak GiB, seconds) or ("OOM", None, seconds)."""
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    base = torch.cuda.memory_allocated()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    try:
        out = fn()
        torch.cuda.synchronize()
        return out, (torch.cuda.max_memory_allocated() - base) / GiB, time.time() - t0
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return "OOM", None, time.time() - t0


@torch.no_grad()
def full_context_icl(document, query, model, tok, answer_ids):
    x = tok(format_prompt(document, query), return_tensors="pt").input_ids.to(model.device)
    return two_way(model(x).logits[:, -1, :], answer_ids).cpu()


def scaling_pool(ds):
    return [it["parametric_memory"].strip() for it in ds[-3000:] if len(it.get("parametric_memory", "").strip()) > 50]


def memory_sweep(tok, model, out, lengths=DEFAULT_LENGTHS, docs_per_length=1, seed=42):
    """Memory/time of AIR, full-context ICL and split-window ICL per length. Resumes from memory_scaling.json."""
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    path = out / "memory_scaling.json"
    rows = json.load(open(path)) if path.exists() else []
    done = {(r["target_tokens"], r["doc"]) for r in rows}
    answer_ids, cfg = answer_token_ids(tok), lora_config(r=2)
    ds = load_conflictqa()
    pool, items = scaling_pool(ds), make_items(ds, max(docs_per_length, 2), seed)
    for L in lengths:
        for d in range(docs_per_length):
            if (L, d) in done:
                continue
            item = items[d % len(items)]
            doc = build_document(item, pool, L, tok, random.Random(seed * 1000 + L + d))
            row = {"target_tokens": L, "doc_tokens": len(tok.encode(doc, add_special_tokens=False)), "doc": d}

            res, mem, sec = measure(lambda: air_beliefs(doc, item["query"], model, tok, cfg, answer_ids))
            if isinstance(res, str):
                raise RuntimeError("AIR ran out of memory, which should not happen; the LoRA wrapper may still be attached")
            model = res[3]
            row.update(air_chunks=len(res[2]), air_peak_gib=mem, air_seconds=sec, air_oom=False)

            res, mem, sec = measure(lambda: full_context_icl(doc, item["query"], model, tok, answer_ids))
            row.update(full_icl_peak_gib=mem, full_icl_seconds=sec, full_icl_oom=isinstance(res, str))

            res, mem, sec = measure(lambda: icl_split_window(doc, item["query"], model, tok, answer_ids, max_length=10 ** 7))
            row.update(split_icl_peak_gib=mem, split_icl_seconds=sec, split_icl_oom=isinstance(res, str))

            rows.append(row)
            json.dump(rows, open(path, "w"), indent=1)
            fmt = lambda v: "OOM" if v is None else f"{v:6.2f}"
            print(f"L={row['doc_tokens']:>7} | AIR +{fmt(row['air_peak_gib'])} GiB {row['air_seconds']:7.1f}s | "
                  f"full ICL +{fmt(row['full_icl_peak_gib'])} GiB | split ICL +{fmt(row['split_icl_peak_gib'])} GiB")
    plot_memory(rows, out)
    return rows, model


def auc_at_length(tok, model, out, length=8192, n_docs=40, rhos=(0.15, 0.35), seed=42):
    """Conflict-detection AUC on longer documents. Resumes per document from auc_at_<length>.json."""
    from sentence_transformers import SentenceTransformer
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    path = out / f"auc_at_{length}.json"
    state = json.load(open(path)) if path.exists() else {"y": [], "air_confidence_only": [], "icl_split_window": [],
                                                        "air_hybrid": {str(r): [] for r in rhos}}
    answer_ids, cfg = answer_token_ids(tok), lora_config(r=2)
    st = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")
    ds = load_conflictqa()
    pool, items = scaling_pool(ds), make_items(ds, n_docs, seed + 1)
    for i in range(len(state["y"]), len(items)):
        item = items[i]
        doc = build_document(item, pool, length, tok, random.Random(seed * 7919 + i))
        beliefs, confs, chunks, model = air_beliefs(doc, item["query"], model, tok, cfg, answer_ids)
        states, idx = confidence_gate(beliefs, confs)
        lex = [lexical_relevance(item["raw_question"], c, STOPWORDS) for c in chunks]
        e = st.encode([item["raw_question"]] + chunks, normalize_embeddings=True, show_progress_bar=False)
        rel = [0.5 * a + 0.5 * b for a, b in zip(lex, np.clip(e[0] @ e[1:].T, 0, 1))]
        npb = [b.numpy() for b in beliefs]
        for r in rhos:
            state["air_hybrid"][str(r)].append(gated_score(npb, confs, rel, r)[0])
        state["air_confidence_only"].append(max_cross_jsd(states, [confs[j] for j in idx]))
        state["icl_split_window"].append(icl_split_window(doc, item["query"], model, tok, answer_ids, max_length=10 ** 7))
        state["y"].append(item["label"])
        json.dump(state, open(path, "w"), indent=1)
        print(f"  AUC at {length}: {i + 1}/{len(items)} documents ({len(chunks)} chunks)")
    y = np.array(state["y"])
    summary = {"length": length, "n": len(y), "auc_air_confidence_only": roc_auc_score(y, state["air_confidence_only"]),
               "auc_icl_split_window": roc_auc_score(y, state["icl_split_window"])}
    for r in rhos:
        summary[f"auc_air_hybrid_rho_{r}"] = roc_auc_score(y, state["air_hybrid"][str(r)])
    state["summary"] = summary
    json.dump(state, open(path, "w"), indent=1)
    print(json.dumps(summary, indent=1))
    return summary, model


def plot_memory(rows, out):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        return
    fig, ax = plt.subplots(1, 2, figsize=(7, 2.6))
    for key, label, style in [("air", "AIR", "-o"), ("full_icl", "Full-context ICL", "-s"), ("split_icl", "Split-window ICL", "-^")]:
        pts = sorted((r["doc_tokens"], r[f"{key}_peak_gib"], r[f"{key}_seconds"]) for r in rows if r[f"{key}_peak_gib"] is not None)
        if pts:
            xs, ms, ts = zip(*pts)
            ax[0].plot(xs, ms, style, label=label, ms=3)
            ax[1].plot(xs, ts, style, label=label, ms=3)
        oom = [r["doc_tokens"] for r in rows if r[f"{key}_oom"]]
        if oom:
            ax[0].axvline(min(oom), ls=":", lw=0.8, color="grey")
            ax[0].text(min(oom), ax[0].get_ylim()[1] * 0.9, f"{label} OOM", fontsize=6, rotation=90, va="top")
    for a, ylab in zip(ax, ["Additional peak memory (GiB)", "Wall-clock time (s)"]):
        a.set_xscale("log", base=2)
        a.set_xlabel("Document length (tokens)")
        a.set_ylabel(ylab)
        a.grid(alpha=0.3)
    ax[1].set_yscale("log")
    ax[0].legend(fontsize=6)
    fig.tight_layout()
    fig.savefig(Path(out) / "memory_scaling.pdf")
    plt.close(fig)
    print(f"saved {Path(out) / 'memory_scaling.pdf'}")

### `experiments/baselines_full_document.py`
Experiment 1.2: stronger baselines that see the WHOLE document, on the paper's exact test documents.

In [ ]:
import argparse
import json
import sys
from itertools import permutations
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import roc_auc_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer


JUDGE_PROMPT = (
    "Read the document below. Then answer the question with a single word, Yes or No.\n\n"
    "Document:\n{doc}\n\n"
    "Question: Does the document contain two passages that give conflicting answers to the question \"{q}\"?"
)


@torch.no_grad()
def llm_judge(records, model_name, load_4bit):
    tok, model = load_model(model_name, load_4bit)
    answer_ids = answer_token_ids(tok, variants=True)
    scores = []
    for i, r in enumerate(records):
        msgs = [{"role": "user", "content": JUDGE_PROMPT.format(doc=r["document"], q=r["raw_question"])}]
        if tok.chat_template:
            text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        else:
            text = msgs[0]["content"] + "\nAnswer:"
        x = tok(text, return_tensors="pt", add_special_tokens=not tok.chat_template).input_ids.to(model.device)
        scores.append(float(two_way(model(x).logits[:, -1, :], answer_ids)[0]))
        if (i + 1) % 50 == 0:
            print(f"  judge {model_name}: {i + 1}/{len(records)}")
    del model
    torch.cuda.empty_cache()
    return scores


class NLI:
    def __init__(self, name):
        self.tok = AutoTokenizer.from_pretrained(name)
        self.model = AutoModelForSequenceClassification.from_pretrained(name).to("cuda").eval()
        self.contra = [k for k, v in self.model.config.id2label.items() if "contradict" in v.lower()][0]

    @torch.no_grad()
    def contradiction(self, premise, hypothesis):
        enc = self.tok(premise, hypothesis, return_tensors="pt", truncation=True, max_length=512).to("cuda")
        return float(torch.softmax(self.model(**enc).logits, dim=-1)[0, self.contra])


def retrieve_nli(records, nli, rels, top_k):
    scores = []
    for r, rel in zip(records, rels):
        top = sorted(range(len(r["chunks"])), key=lambda i: rel[i], reverse=True)[:top_k]
        scores.append(max(nli.contradiction(r["chunks"][a], r["chunks"][b]) for a, b in permutations(top, 2)))
    return scores


def oracle_nli(records, nli):
    return [nli.contradiction(f"Question: {r['query']} Evidence: {r['doc_a']}", r["doc_b"]) for r in records]


DEFAULT_NLI = ["cross-encoder/nli-deberta-v3-small", "cross-encoder/nli-deberta-v3-base",
               "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"]


def run_baselines(benchmark="conflictqa", air_run=None, judge_models=("google/gemma-2-2b-it",), load_4bit=False,
                  nli_models=tuple(DEFAULT_NLI), top_k=3, n_boot=2000, out=None, tag=""):
    """Score whole-document baselines and compare them with AIR on the same documents. Returns the report dict."""
    out = Path(out or ROOT / "results" / "baselines")
    out.mkdir(parents=True, exist_ok=True)

    if air_run:
        run = json.load(open(air_run, encoding="utf-8"))
        cfg = run["config"]
        docs = build(cfg["benchmark"], cfg["seed"])
        if cfg.get("n_docs") and cfg["n_docs"] < len(docs):
            docs = balanced_subset(docs, cfg["n_docs"])
        for rec, d in zip(run["records"], docs):
            rec.update(document=d["document"], doc_a=d["doc_a"], doc_b=d["doc_b"])
    else:
        run = load_notebook_run(benchmark)
    records = run["records"]
    y = np.array([r["label"] for r in records])
    rels = relevance(records, benchmark)

    scores = {}
    for m in judge_models:
        scores[f"LLM judge ({m.split('/')[-1]})"] = llm_judge(records, m, load_4bit)
    for m in nli_models:
        nli = NLI(m)
        scores[f"Retrieve-then-NLI top-{top_k} ({m.split('/')[-1]})"] = retrieve_nli(records, nli, rels["hybrid"], top_k)
        scores[f"Oracle-pair NLI ({m.split('/')[-1]})"] = oracle_nli(records, nli)
        del nli
        torch.cuda.empty_cache()

    # AIR on the same documents
    docs = gate_docs(records)
    air = {f"AIR hybrid (rho={PAPER_RHO['hybrid']})": scores_for(docs, rels["hybrid"], PAPER_RHO["hybrid"])[0],
           "AIR hybrid (CV rho)": cv_auc(y, docs, rels["hybrid"], 20)[3]}
    if all("icl_split_window" in r for r in records):
        scores["ICL split-window"] = [r["icl_split_window"] for r in records]

    report = {"benchmark": benchmark, "n": int(len(y)), "air_source": run["config"], "baselines": {}, "air": {}}
    print(f"\n=== {benchmark} (N={len(y)}) ===")
    for name, s in air.items():
        report["air"][name] = {"auc": float(roc_auc_score(y, s))}
        print(f"{name:<58} AUC {roc_auc_score(y, s):.3f}")
    for name, s in scores.items():
        s = np.asarray(s, dtype=float)
        entry = {"auc": float(roc_auc_score(y, s)), "scores": s.tolist()}
        for air_name, air_s in air.items():
            ci, dci, p = bootstrap(y, np.asarray(air_s), s, n_boot)
            entry[f"{air_name} minus baseline"] = {"diff_ci": dci, "p": p}
        report["baselines"][name] = entry
        cv = entry["AIR hybrid (CV rho) minus baseline"]
        print(f"{name:<58} AUC {entry['auc']:.3f} | AIR(CV) - baseline CI [{cv['diff_ci'][0]:+.3f}, {cv['diff_ci'][1]:+.3f}] p={cv['p']:.3f}")
    path = out / f"{benchmark}{tag}_baselines.json"
    json.dump(report, open(path, "w", encoding="utf-8"), indent=1)
    print(f"saved {path}")
    return report

### `experiments/rank_ablation.py`
Experiment 1.6: Micro-LoRA rank and distillation-steps ablation with the FINAL gate, on ConflictQA.

In [ ]:
import argparse
from pathlib import Path


def run_rank_ablation(n_docs=100, ranks=(1, 2, 4, 8, 16), steps_grid=(3, 5, 20), model=DEFAULT_MODEL,
                      out_dir=None, tok_model=None):
    """Run every (rank, steps) configuration on the same balanced ConflictQA subset; returns the loaded model."""
    out_dir = out_dir or str(ROOT / "results" / "rank_ablation")
    configs = [(r, 10) for r in ranks] + [(2, s) for s in steps_grid if s != 10]
    for rank, steps in configs:
        print(f"\n### rank={rank} steps={steps}")
        out, tok_model = run("conflictqa", 42, model, rank, steps, n_docs, skip_icl=True, out_dir=out_dir, tok_model=tok_model)
        print(f"saved {out}")
    return tok_model

### `experiments/summarize.py`
Turn experiment runs into paper-ready tables (CPU only).

In [ ]:
import argparse
import json
from collections import defaultdict
from pathlib import Path

import numpy as np


KEY_METRICS = [("AIR confidence-only", "auc"), ("ICL split-window", "auc"), ("AIR hybrid", "auc"),
               ("AIR hybrid", "cv_auc_mean"), ("AIR lexical", "auc"), ("AIR semantic", "auc")]


def load_runs(paths):
    runs = []
    for p in paths:
        run = json.load(open(p, encoding="utf-8"))
        if len(run.get("records", [])) < run["config"].get("n_docs", 0):
            print(f"skipping {p.name}: incomplete ({len(run['records'])}/{run['config']['n_docs']} documents)")
            continue
        run["config"]["file"] = p.name
        runs.append(run)
    return runs


def fmt(v, digits=3):
    return "--" if v is None else f"{v:.{digits}f}"


def rank_table(rank_dir=None, out=None):
    """Rank / steps ablation table from rank_ablation.py runs; writes rank_ablation.json and rank_table.tex."""
    rank_dir = Path(rank_dir or ROOT / "results" / "rank_ablation")
    out = Path(out or ROOT / "results" / "summary")
    out.mkdir(parents=True, exist_ok=True)
    rows = []
    for run in load_runs(sorted(rank_dir.glob("*.json"))):
        c, recs = run["config"], run["records"]
        m = score_run(recs, "conflictqa", n_boot=200)["methods"]
        chunks = sum(r["num_chunks"] for r in recs)
        rows.append({"rank": c["rank"], "alpha": c["alpha"], "steps": c["steps"], "adapter_mb": c["adapter_bytes"] / 2 ** 20,
                     "sec_per_chunk": sum(r["air_seconds"] for r in recs) / chunks, "n": len(recs),
                     "auc_conf_only": m["AIR confidence-only"]["auc"], "auc_hybrid": m["AIR hybrid"]["auc"],
                     "f1_hybrid": m["AIR hybrid"]["f1"], "cv_auc_hybrid": m["AIR hybrid"]["cv_auc_mean"]})
    rows.sort(key=lambda r: (r["steps"] != 10, r["rank"], r["steps"]))
    md = ["## Rank / steps ablation (ConflictQA)", "",
          "| r | alpha | steps | adapter MB | s/chunk | AUC conf-only | AUC hybrid (rho=0.35) | F1 hybrid | CV AUC hybrid |",
          "|---|---|---|---|---|---|---|---|---|"]
    md += [f"| {r['rank']} | {r['alpha']} | {r['steps']} | {r['adapter_mb']:.2f} | {r['sec_per_chunk']:.2f} | {r['auc_conf_only']:.3f} | "
           f"{r['auc_hybrid']:.3f} | {r['f1_hybrid']:.3f} | {r['cv_auc_hybrid']:.3f} |" for r in rows]
    latex = [r"\begin{tabular}{c c c r c c c}", r"\toprule",
             r"\textbf{$r$} & \textbf{$\alpha$} & \textbf{Steps} & \textbf{Adapter} & \textbf{s/chunk} & \textbf{AUC} & \textbf{F1} \\",
             r"\midrule"]
    latex += [f"{r['rank']} & {r['alpha']} & {r['steps']} & {r['adapter_mb']:.2f}" + r"\,MB" + f" & {r['sec_per_chunk']:.2f} & "
              f"{r['auc_hybrid']:.3f} & {r['f1_hybrid']:.3f} " + r"\\" for r in rows]
    latex += [r"\bottomrule", r"\end{tabular}"]
    (out / "rank_table.tex").write_text("\n".join(latex), encoding="utf-8")
    (out / "rank_table.md").write_text("\n".join(md), encoding="utf-8")
    json.dump(rows, open(out / "rank_ablation.json", "w"), indent=1)
    print("\n".join(md))
    print(f"\nLaTeX table written to {out / 'rank_table.tex'}")
    return rows


def summarize_runs(runs_dir=None, include_paper=True, n_boot=2000, out=None):
    """Score every run (plus the paper's seed-42 results), aggregate over seeds, write summary.md / summary.json."""
    runs_dir = Path(runs_dir or ROOT / "results" / "runs")
    out = Path(out or ROOT / "results" / "summary")
    out.mkdir(parents=True, exist_ok=True)
    runs = load_runs(sorted(runs_dir.glob("*.json"))) if runs_dir.exists() else []
    if include_paper:
        for b in ["conflictqa", "strategyqa"]:
            runs.append(load_notebook_run(b))
    if not runs:
        raise RuntimeError("no runs found")

    scored = []
    for run in runs:
        c = run["config"]
        print(f"scoring {c['benchmark']} seed={c['seed']} model={c['model']} r={c['rank']} steps={c['steps']} ...")
        scored.append({"config": c, "metrics": score_run(run["records"], c["benchmark"], n_boot=n_boot)})

    groups = defaultdict(list)
    for s in scored:
        c = s["config"]
        groups[(c["benchmark"], c["model"], c["rank"], c["steps"], s["metrics"]["n"])].append(s)

    md = ["## Mean over seeds (ROC-AUC)", "",
          "| benchmark | model | r | steps | seeds | conf-only | ICL | hybrid (rho=0.35) | hybrid (CV rho) | lexical | semantic |",
          "|---|---|---|---|---|---|---|---|---|---|---|"]
    summary = []
    for (bench, model, rank, steps, n), items in sorted(groups.items()):
        row = {"benchmark": bench, "model": model, "rank": rank, "steps": steps, "n": n,
               "seeds": sorted(i["config"]["seed"] for i in items)}
        cells = []
        for meth, key in KEY_METRICS:
            vals = [i["metrics"]["methods"][meth][key] for i in items if meth in i["metrics"]["methods"]]
            row[f"{meth} {key}"] = {"mean": float(np.mean(vals)), "std": float(np.std(vals))} if vals else None
            cells.append("--" if not vals else (f"{np.mean(vals):.3f}" + (f" ± {np.std(vals):.3f}" if len(vals) > 1 else "")))
        summary.append(row)
        md.append(f"| {bench} | {model.split('/')[-1]} | {rank} | {steps} | {', '.join(map(str, row['seeds']))} | " + " | ".join(cells) + " |")

    md += ["", "## Per run: hybrid gate detail", "",
           "| benchmark | seed | model | AUC [95% CI] | p vs ICL | CV AUC | CV p vs ICL | abstained cons/conf |",
           "|---|---|---|---|---|---|---|---|"]
    for s in scored:
        c, h = s["config"], s["metrics"]["methods"]["AIR hybrid"]
        ci = h.get("ci")
        md.append(f"| {c['benchmark']} | {c['seed']} | {c['model'].split('/')[-1]} | {h['auc']:.3f}"
                  + (f" [{ci[0]:.3f}, {ci[1]:.3f}]" if ci else "") + f" | {fmt(h.get('p_vs_icl'))} | {h['cv_auc_mean']:.3f} | "
                  f"{fmt(h.get('cv_p_vs_icl'))} | {h['abstained_consistent']}/{h['abstained_conflict']} |")

    json.dump({"groups": summary, "runs": scored}, open(out / "summary.json", "w", encoding="utf-8"), indent=1)
    (out / "summary.md").write_text("\n".join(md), encoding="utf-8")
    print("\n" + "\n".join(md))
    print(f"\nwritten to {out}")
    return summary

## Crash guard and shared model

In [ ]:
STATUS = {}
GPU_EXPERIMENTS = {k for k in RUN if k != "summary"}
_SHARED = {"tok_model": None}


def log(msg):
    stamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{stamp}] {msg}")
    with open(LOG, "a", encoding="utf-8") as f:
        f.write(f"[{stamp}] {msg}\n")


def gemma():
    """The shared Gemma model, loaded once and reused across experiments."""
    if _SHARED["tok_model"] is None:
        _SHARED["tok_model"] = load_model(DEFAULT_MODEL)
    return _SHARED["tok_model"]


def release_model():
    """Drop the shared model (after a failure it may still carry a LoRA wrapper) and free GPU memory."""
    _SHARED["tok_model"] = None
    import gc
    gc.collect()
    torch.cuda.empty_cache()


def guarded(name, fn):
    """Run one experiment; on any error, log the traceback and continue with the next experiment."""
    if not RUN.get(name):
        STATUS[name] = "skipped (disabled)"
        return None
    if name in GPU_EXPERIMENTS and not HAS_GPU:
        STATUS[name] = "skipped (no GPU)"
        return None
    log(f"START {name}")
    t0 = time.time()
    try:
        result = fn()
        STATUS[name] = f"done ({(time.time() - t0) / 3600:.2f} h)"
        log(f"DONE  {name} in {(time.time() - t0) / 3600:.2f} h")
        return result
    except Exception:
        STATUS[name] = f"FAILED after {(time.time() - t0) / 3600:.2f} h - see {LOG.name}"
        log(f"FAILED {name}\n{traceback.format_exc()}")
        release_model()
        return None

## Experiments

### 1. Memory and time vs. document length
Answers: *“the O(1) / web-scale claim is never measured.”* Output: `results/memory_scaling/`.

In [ ]:
def _memory():
    tok, model = gemma()
    _, model = memory_sweep(tok, model, ROOT / "results" / "memory_scaling", MEMORY_LENGTHS, 1, 42)
    _SHARED["tok_model"] = (tok, model)

guarded("memory_scaling", _memory)

### 2. Whole-document baselines on ConflictQA
Answers: *“the baselines are weak; nothing reads the whole document.”* Gemma as a whole-document judge, retrieve-then-NLI and three NLI models, each tested against AIR on the paper's documents. Output: `results/baselines/`.

In [ ]:
guarded("baselines_conflictqa", lambda: run_baselines("conflictqa", out=ROOT / "results" / "baselines"))

### 3. Extra seeds on ConflictQA
Answers: *“single seed.”* New items, distractors and LoRA initialization per seed. Output: `results/runs/`.

In [ ]:
def _seeds_conflictqa():
    for seed in EXTRA_SEEDS:
        _, _SHARED["tok_model"] = run("conflictqa", seed, tok_model=gemma(), out_dir=ROOT / "results" / "runs")

guarded("seeds_conflictqa", _seeds_conflictqa)

### 4. De-biased StrategyQA
Answers: *“StrategyQA leaks the label to the gate.”* Both target passages restate the question in both classes. Runs AIR + ICL, then the whole-document baselines on the same documents.

In [ ]:
def _debiased():
    out, _SHARED["tok_model"] = run("strategyqa_debiased", 42, tok_model=gemma(), out_dir=ROOT / "results" / "runs")
    run_baselines("strategyqa_debiased", air_run=out, out=ROOT / "results" / "baselines")

guarded("strategyqa_debiased", _debiased)

### 5. Accuracy on longer documents
Answers: *“does accuracy hold beyond 2k tokens?”* Output: `results/memory_scaling/auc_at_8192.json`.

In [ ]:
def _auc_long():
    tok, model = gemma()
    _, model = auc_at_length(tok, model, ROOT / "results" / "memory_scaling", AUC_LENGTH, AUC_DOCS)
    _SHARED["tok_model"] = (tok, model)

guarded("auc_long_documents", _auc_long)

### 6. Extra seeds on StrategyQA
Same as 3, for the original StrategyQA benchmark.

In [ ]:
def _seeds_strategyqa():
    for seed in EXTRA_SEEDS:
        _, _SHARED["tok_model"] = run("strategyqa", seed, tok_model=gemma(), out_dir=ROOT / "results" / "runs")

guarded("seeds_strategyqa", _seeds_strategyqa)

### 7. Rank and distillation-steps ablation
Answers: *“the rank ablation has no numbers.”* Uses the final gate. Output: `results/rank_ablation/`.

In [ ]:
def _rank():
    _SHARED["tok_model"] = run_rank_ablation(RANK_N_DOCS, out_dir=str(ROOT / "results" / "rank_ablation"), tok_model=gemma())

guarded("rank_ablation", _rank)

### 8. Large long-context judge
A 7B instruction model in 4-bit reads each whole document. Frees the shared Gemma model first.

In [ ]:
def _large_judge():
    release_model()
    run_baselines("conflictqa", judge_models=[LARGE_JUDGE], load_4bit=True, nli_models=[],
                  out=ROOT / "results" / "baselines", tag="_large_judge")

guarded("baselines_large_judge", _large_judge)

### 9. Second base model
Answers: *“only one 2B model.”* AIR + ICL on ConflictQA with another model family.

In [ ]:
def _second_model():
    release_model()
    out, tok_model = run("conflictqa", 42, model_name=SECOND_MODEL, out_dir=ROOT / "results" / "runs")
    del tok_model
    release_model()

guarded("second_model", _second_model)

### 10. Paper-ready summary
Mean ± std over seeds for every method, per-run significance tests, and the rank table (also as LaTeX). Output: `results/summary/`.

In [ ]:
def _summary():
    summarize_runs(ROOT / "results" / "runs", include_paper=True, out=ROOT / "results" / "summary")
    if any((ROOT / "results" / "rank_ablation").glob("*.json")):
        rank_table(ROOT / "results" / "rank_ablation", ROOT / "results" / "summary")

guarded("summary", _summary)

## Status report

In [ ]:
log("ALL EXPERIMENTS FINISHED")
width = max(len(k) for k in STATUS)
report = "\n".join(f"{k:<{width}}  {v}" for k, v in STATUS.items())
print(report)
(ROOT / "results" / "run_all_status.txt").write_text(report + "\n", encoding="utf-8")
print(f"\nlog: {LOG}\nsummary tables: {ROOT / 'results' / 'summary'}")

if SHUTDOWN_WHEN_DONE:
    log("shutting down")
    os.system("shutdown /s /t 60" if os.name == "nt" else "shutdown -h now")